I will be predicting whether the 'Man of the match' winner came from a given team after a match has ended. A match contributes two teams rows, one labeled 1 and the other 0. The two rows are linked observation, so later splitting and evaluation happen at the match level.

In [1]:
import numpy as np
import pandas as pd

# Relative path: the CSV sits next to the notebook, so this runs on any machine.
DATASET_PATH = "world_cup_team_statistics.csv"
TARGET = "Man of the Match"

df = pd.read_csv(DATASET_PATH)

print(df.shape) #show rows and columns

print(df["match_id"].nunique()) #show number of matches

print(df["year"].value_counts().sort_index()) #tournament years and number of matches played in each year

print("Target counts:")
print(df[TARGET].value_counts().sort_index())

print(df.head())

(256, 45)
128
year
2018    128
2022    128
Name: count, dtype: int64
Target counts:
Man of the Match
0    128
1    128
Name: count, dtype: int64
         Date          Team      Opponent  Goal Scored  Ball Possession %  \
0  2018-06-14        Russia  Saudi Arabia            5               40.0   
1  2018-06-14  Saudi Arabia        Russia            0               60.0   
2  2018-06-15         Egypt       Uruguay            0               43.0   
3  2018-06-15       Uruguay         Egypt            1               57.0   
4  2018-06-15          Iran       Morocco            1               36.0   

   Attempts  On-Target  Off-Target  Blocked  Corners  ...  \
0        13          7           3        3        6  ...   
1         6          0           3        3        2  ...   
2         8          3           3        2        0  ...   
3        14          4           6        4        5  ...   
4         8          2           5        1        2  ...   

   award_winner_player_id

The target is balanced by construction, but the effective sample size is 128 matches rather than 256 independent decisions. passes_completed, possession_in_contest_pct, incomplete_attempts, and other supplemental columns are available only for 2022. The 2022 statistics_method says possession was normalized to exclude an in-contest share and pass accuracy was computed.
I will use shared match-stat columns for the main comparison and report year-specific results later.

In [5]:
assert df["match_id"].notna().all() # check if there are any missing values in the match_id column
assert df["Team"].notna().all() # check if there are any missing values in the Team column
assert df["Opponent"].notna().all() # check if there are any missing values in the Opponent column
assert df[TARGET].notna().all() # check if there are any missing values in the Man of the Match column
assert set(df[TARGET].unique()) == {0, 1} # check if the Man of the Match column only contains 0 and 1

match_check = df.groupby("match_id").agg(
    rows=(TARGET, "size"), 
    positive_labels=(TARGET, "sum"),
    years=("year", "nunique"),
) 

assert (match_check["rows"]  == 2).all() #check if there are exactly 2 rows for each match_id
assert (match_check["positive_labels"]  == 1).all() #check if there is exactly 1 positive label for each match_id
assert (match_check["years"]  == 1).all() #check if there is exactly 1 unique year for each match_id
assert not df.duplicated(subset=["match_id", "Team"]).any() #check if there are any duplicate rows for the same match_id and Team combination

for _, pair in df.groupby("match_id"):
    a, b = pair.iloc[0], pair.iloc[1]
    assert a["Team"] == b["Opponent"]
    assert b["Team"] == a["Opponent"]

print("All matches have two reciprocal teams and one positive label for Man of the Match.")

All matches have two reciprocal teams and one positive label for Man of the Match.


Preprocessing for inspecting missingness and source differences

In [8]:
missing = df.isna().sum().sort_values(ascending=False)
display(missing[missing > 0].to_frame("missing_rows"))

source_columns = [
    "award_status", "statistics_source", "award_source", "statistics_method"
]
for column in source_columns:
    print(f"\n{column}")
    display(pd.crosstab(df["year"], df[column]))

display(
    df.groupby("year")[[
        "Ball Possession %", "Pass Accuracy %", "Passes",
        "Distance Covered (Kms)", "Attempts"
    ]].agg(["mean", "min", "max"]).round(2)
)


,missing_rows
Own goal Time,242
award_winner_player,128
passes_completed,128
on_target_shot_outcomes,128
on_target_fifa_summary,128
incomplete_attempts,128
possession_in_contest_pct,128
fifa_report_url,128
award_winner_player_id,128
1st Goal,73



award_status


award_status,reported,reported_online
year,,
2018,128,0
2022,0,128



statistics_source


statistics_source,FIFA 2018 Statistics.csv,"FIFA post-match report; supplemental match CSV for corners, offsides and fouls"
year,,
2018,128,0
2022,0,128



award_source


award_source,FIFA 2018 Statistics.csv (user supplied),https://www.soccer-blogger.com/2022/12/17/2022-football-world-cup-man-of-the-match-winners-list-of-motm-awards-at-fifa-wc-2022/
year,,
2018,128,0
2022,0,128



statistics_method


statistics_method,FIFA normalized possession excluding in-contest share; computed pass accuracy,Supplied 2018 statistics
year,,
2018,0,128
2022,128,0


Ball Possession %               Pass Accuracy %                Passes  \
                  mean    min    max            mean    min    max    mean   
year                                                                         
2018              50.0  25.00  75.00           82.55  67.00  94.00  462.65   
2022              50.0  15.82  84.18           84.11  65.38  94.53  500.80   

                Distance Covered (Kms)               Attempts          
      min   max                   mean    min    max     mean min max  
year                                                                   
2018  189  1137                 106.66   80.0  148.0    12.59   3  26  
2022  224  1070                 115.25  102.3  148.2    11.21   0  32

The target is balanced, I will use shared match-stat columns for the main comparison and report year-specific results later.